# 06 — Grad-CAM analysis

Inference-only explanations of the **verified Phase 4 models**, using the unchanged frozen manifest and model-specific preprocessing. Individual maps explain signed binary logits; equal and frozen-weight maps are **heatmap fusion**, not direct ensemble-score Grad-CAM. No retraining, tuning or resplitting occurs.

All execution flags default to False. No private checkpoint, image or prediction file is read unless analysis/inspection is explicitly enabled. Image-linked artifacts and displayed outputs are private: retain them under ignored artifact storage, never commit executed outputs or send them to Gemini.

## 1. Independent Kaggle / Colab / local setup
Use the saved Phase 3 session, private original Phase 4 run (including withheld CSVs), original manifest and all selected checkpoint recovery files. The public documentation archive alone is insufficient. Set the paths below to your saved inputs; Kaggle working outputs must be saved/exported privately before the session ends.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import copy
import json

RUN_PLATFORM_SETUP = False
CLONE_REPOSITORY = False
INSTALL_DEPENDENCIES = False
RUN_GRADCAM = False
ALLOW_TEST_ANALYSIS = False
INSPECT_SAVED_PREDICTIONS = False
SHOW_PRIVATE_PANELS = False

PLATFORM = "kaggle" if (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").is_dir()) else "colab" if Path("/content").is_dir() else "local"
default_repo = "/kaggle/working/glaucoma_detection" if PLATFORM == "kaggle" else "/content/glaucoma_detection" if PLATFORM == "colab" else str(Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent)
REPO_ROOT = Path(os.environ.get("PROJECT_REPO", default_repo))
REPO_URL = "https://github.com/VDJ2824/glaucoma_detection.git"
REPO_REF = "main"  # prefer a reviewed commit for reproducibility
BUNDLE_ROOT = Path("/kaggle/input/your-private-training-run")  # EDIT: run containing checkpoints/manifests/environment
if PLATFORM == "kaggle":
    default_session = BUNDLE_ROOT / "environment/session.json"
    default_output = Path("/kaggle/working/artifacts")
elif PLATFORM == "colab":
    default_output = Path("/content/drive/MyDrive/glaucoma_detection/artifacts")
    default_session = default_output / "runs/development/environment/session.json"
else:
    default_output = REPO_ROOT / "artifacts"
    default_session = default_output / "runs/development/environment/session.json"
SESSION_FILE = os.environ.get("GLAUCOMA_SESSION", str(default_session))
MANIFEST_CSV = BUNDLE_ROOT / "manifests/acrima_metadata.csv" if PLATFORM == "kaggle" else None
CHECKPOINT_ROOT = BUNDLE_ROOT / "checkpoints/core" if PLATFORM == "kaggle" else None
# Optional per-model bases containing <model>/<stage>/; leave None for a common base/session setting.
CHECKPOINT_ROOTS = None
# To decode original Kaggle manifest paths on another machine, set the CURRENT flat image directory.
# Keep the original CSV unchanged; this affects only in-memory physical path lookup.
FLAT_IMAGE_ROOT = None
OUTPUT_ROOT = default_output
EVALUATION_NAME = "acrima_phase4_v1"

# Root of the COMPLETE PRIVATE Phase 4 evaluation run, not docs/experiments.
EVALUATION_DIRECTORY = BUNDLE_ROOT / "phase4/acrima_phase4_v1" if PLATFORM == "kaggle" else OUTPUT_ROOT / "phase4/acrima_phase4_v1"
SPLIT = "validation"  # "test" requires ALLOW_TEST_ANALYSIS and test_complete.json
EXPLANATION_NAME = "acrima_gradcam_validation_v1"  # fresh output name per analysis
HEATMAP_ROOT = None  # optional PRIVATE Drive/Kaggle output root; never docs/ or dataset storage
PER_CATEGORY = 2
SELECTION_METHOD = "weighted_soft_voting"  # correct/incorrect relative to this evaluated method
TARGET_CLASSES = (0, 1)  # same target for all maps within each panel


In [ ]:
if RUN_PLATFORM_SETUP:
    if not (3, 10) <= sys.version_info[:2] < (3, 14):
        raise RuntimeError("Use Python 3.10–3.13.")
    if PLATFORM == "colab":
        from google.colab import drive
        drive.mount("/content/drive")
    if CLONE_REPOSITORY:
        if REPO_ROOT.exists():
            raise FileExistsError("Existing clone preserved; inspect it rather than resetting it.")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF], check=True)
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT/"requirements-training.txt")], cwd=REPO_ROOT, check=True)
        raise RuntimeError("Restart the kernel, disable installation, then rerun.")
if not (REPO_ROOT/"src/glaucoma").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the existing project clone.")
sys.path.insert(0, str(REPO_ROOT/"src"))


## 2. Load the saved session and import reusable modules
Original preprocessing is reused exactly once per image/model, with no augmentation. Physical checkpoint/image directories may move; content hashes, graph/contract identity, frozen manifest and inference settings must still agree. Set FLAT_IMAGE_ROOT only for physical lookup; do not rewrite the manifest. For Colab, use mounted Drive for HEATMAP_ROOT. Setup/import cells do not construct models or generate explanations.


In [ ]:
from glaucoma.utils.notebook_runtime import load_session
from glaucoma.ensemble.predictions import MEMBERS
from glaucoma.explainability.workflow import load_analysis_context, select_examples, run_explanations

cfg = copy.deepcopy(load_session(SESSION_FILE))
if HEATMAP_ROOT is not None:
    cfg["artifacts"]["heatmaps"] = str(HEATMAP_ROOT)
manifest_path = Path(MANIFEST_CSV or cfg["metadata"]["output"])
checkpoint_roots = CHECKPOINT_ROOTS or ({name: str(CHECKPOINT_ROOT) for name in MEMBERS} if CHECKPOINT_ROOT is not None else None)
selection_records = {
    "mobilenet_v2": REPO_ROOT/"docs/experiments/mobilenet_v2_kaggle_2026-11-26/metrics/selection.json",
    "efficientnet_b0": REPO_ROOT/"docs/experiments/efficientnet_b0_kaggle_2026-11-27/metrics/selection.json",
    "resnet50": REPO_ROOT/"docs/experiments/resnet50_kaggle_2026-11-27/metrics/selection.json",
}
print("Private Phase 4 run:", EVALUATION_DIRECTORY)
print("Manifest (preserved):", manifest_path)
print("Private heatmap root:", cfg["artifacts"].get("heatmaps"))


## 3. Deterministic correct / incorrect / disagreement examples
Selection sorts saved predictions by dataset and image ID, takes up to PER_CATEGORY from each category, and deduplicates overlapping examples. Missing categories are reported, never fabricated. Correctness refers to SELECTION_METHOD at the frozen threshold. Disagreement means at least one core classifier disagrees with another. Inspection reads private CSVs/manifest only; it does not decode images or generate new predictions.

For test explanations set SPLIT="test" and ALLOW_TEST_ANALYSIS=True **only after Phase 4 test completion**. All test-set examples are post-hoc analysis: do not use them to change models, preprocessing, weights or threshold. The test CSV has no original Phase 4 cryptographic seal; its source hash is recorded and explained-image scores are checked against the recovered models.


In [ ]:
if INSPECT_SAVED_PREDICTIONS:
    aligned, frozen, provenance = load_analysis_context(cfg, manifest_path,
        EVALUATION_DIRECTORY, SPLIT, allow_test=ALLOW_TEST_ANALYSIS,
        flat_image_root=FLAT_IMAGE_ROOT)
    selected, selection_summary = select_examples(aligned, frozen["weights"], frozen["threshold"],
        method=SELECTION_METHOD, per_category=PER_CATEGORY)
    print(json.dumps(selection_summary, indent=2))
    print(selected.to_string(index=False))  # PRIVATE image-linked output
else:
    print("Saved prediction inspection disabled.")


## 4. Individual Grad-CAM and ensemble heatmap fusion
Enable RUN_GRADCAM explicitly. Models load sequentially through the existing checkpoint selection/recovery utilities; byte hashes, architecture graphs and training contracts must match Phase 4. The connected nested-backbone probes use MobileNetV2 `Conv_1`, EfficientNetB0 `top_conv` and ResNet50 `conv5_block3_3_conv`. Probe and head always run with training=False; no weights/trainability are changed.

For p(glaucoma)=sigmoid(z), class 1 maps explain z and class 0 maps explain -z. Class 0 is **not** 1 minus the class 1 heatmap. Channel weights are spatially averaged gradients; the weighted activation sum is ReLU-clipped and max-normalized. Missing gradients and nonfinite maps fail explicitly. Zero gradients / no positive activation yield a zero map and explicit status; fusion is unavailable if any member is noninformative, rather than silently redistributing weights.

Both fusion strategies use the same target class and three normalized maps, interpolate them onto the 224×224 canvas, apply equal or the original Phase 4 weights, and normalize the mixture. These are visual fusions of logit maps, **not derivatives of the ensemble probability score**. Padding is cropped with the original letterbox rounding/offset rules before resizing to original geometry and overlaying. Cropping cannot restore fine anatomical detail lost during downsampling.


In [ ]:
analysis_result = None
if RUN_GRADCAM:
    analysis_result = run_explanations(cfg, manifest_path, EVALUATION_DIRECTORY,
        enabled=True, split=SPLIT, allow_test=ALLOW_TEST_ANALYSIS,
        name=EXPLANATION_NAME, per_category=PER_CATEGORY, method=SELECTION_METHOD,
        target_classes=TARGET_CLASSES, checkpoint_roots=checkpoint_roots,
        expected_records=selection_records, flat_image_root=FLAT_IMAGE_ROOT)
    print(json.dumps(analysis_result, indent=2))
else:
    print("Grad-CAM disabled. No models or images loaded; no heatmaps generated.")


## 5. Private comparative panels and artifacts
Each original image / target class has a six-panel comparison: original (ground truth), three individual overlays (saved score and predicted class), equal fusion and frozen-weight fusion (their scores and predicted classes). A noninformative map is labelled explicitly; no localization is invented. Arrays retain native or original-image spatial coordinates; metadata records target layers, target logit, gradient status, saved/recomputed scores, category, provenance and frozen configuration fingerprint.

Private layout: `<cfg.artifacts.heatmaps>/phase5/<EXPLANATION_NAME>/analysis.json`, `selected_examples.json`, `example_NNN/class_0|class_1/` with `.npy` maps, overlay PNGs, `comparison.png`, `explanation.json`; `complete.json` is written only on success. A failed run stays available for inspection and is not overwritten. Use a fresh name to rerun; do not delete Phase 4 provenance or checkpoints.


In [ ]:
if SHOW_PRIVATE_PANELS:
    if analysis_result is None:
        raise ValueError("Generate a successful private analysis in this session first.")
    from IPython.display import display, Image
    for path in sorted(Path(analysis_result["directory"]).glob("example_*/class_*/comparison.png")):
        display(Image(filename=str(path)))
else:
    print("Private panel display disabled. Keep notebook outputs empty before committing.")


## 6. Interpretation and limitations
Grad-CAM is a coarse, class-targeted sensitivity visualization, not segmentation, a clinical diagnosis or proof of correct anatomical reasoning. Per-map normalization discards absolute magnitude; fusion is heuristic and is not a calibrated explanation or direct ensemble gradient. Empty maps can reflect zero sensitivity, ReLU suppression or padding-only activation. Inspect errors and disagreement cases without changing the evaluated system.

Image-level splitting does not establish patient independence; this remains single-dataset research. Test explanations are retrospective only. No patient IDs are invented, VGG is excluded from core fusion, and images/overlays must remain private until original redistribution terms are verified. See [Phase 5 guide](../docs/phase5-gradcam.md). No retinal results are prefilled here.
